# Building Linear Regression from Scratch: A Mathematical Journey with SymPy

## Implementation: Step by Step

### Setting the environment

In [ ]:
import sympy as sp 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

### Step 1: Define Symbolic Variables

In [ ]:
sp.init_printing(use_latex="mathjax")
m, c, x, y = sp.symbols("m c x y")

### Step 2: Defining the Hypothesis in Symbolic Notation

In [ ]:
hypothesis = m*x + c
print("\nStep 2: Hypothesis Formula")
sp.pprint(hypothesis)

### Step 3: Defining the Cost Function (MSE)

In [ ]:
cost_func = (hypothesis - y)**2
print("\nStep 3: Cost Function (Single Point MSE)")
sp.pprint(cost_func)

### Step 4: Computing Gradients with Calculus

In [ ]:
# Create unevaluated derivative objects for display
grad_m_eqn = sp.Derivative(cost_func, m)
grad_c_eqn = sp.Derivative(cost_func, c)

# Calculate the actual derivative expressions 
grad_m_expr = grad_m_eqn.doit()
grad_c_expr = grad_c_eqn.doit()

print("\nStep 4: Symbolic Gradients (Calculus)", end="\n")
print("Partial Derivative for weight (m):", end="\n")

print("\n\n")
sp.pprint(grad_m_eqn)
print("Evaluates to:")
sp.pprint(grad_m_expr)

print("\nPartial Derivative for Bias (c):")

print("\n\n")
sp.pprint(grad_c_eqn)
print("Evaluates to:")
sp.pprint(grad_c_expr)

### Step 5: Loading Data with Panda

In [ ]:
print("\nStep 5: Loading Data with Pandas", end="\n")
try: 
    df = pd.read_csv("/Volumes/iDrisAI/DeepLearning/Experience_Salary.csv")
    X_train = df['experience'].values
    Y_train = df['salary'].values
    print("\nData successfully loaded from CSV using Pandas.")
except FileNotFoundError:
    print("\nCSV not found")

print("X_train = ", X_train)
print("Y_train = ", Y_train)

### Step 6: Gradient Descent Training


In [ ]:
# Compile symbolic gradients to fast numpy functions once, before training
grad_m_fn = sp.lambdify((m, c, x, y), grad_m_expr, "numpy")
grad_c_fn = sp.lambdify((m, c, x, y), grad_c_expr, "numpy")

def train_linear_regression(data_x, data_y, lr=0.0001, epochs=1000):
    curr_m, curr_c = 0.0, 0.0
    n = len(data_x)
    print(f"\nStep 6: Training (LR={lr}, Epochs={epochs})....")

    for epoch in range(epochs):
        grad_m_vals = grad_m_fn(curr_m, curr_c, data_x, data_y)
        grad_c_vals = grad_c_fn(curr_m, curr_c, data_x, data_y)

        curr_m -= lr * (grad_m_vals.mean())
        curr_c -= lr * (grad_c_vals.mean())

        if epoch % 250 == 0:
            print(f" Epoch {epoch}: m = {curr_m:.4f}, c={curr_c:.4f}")
    return curr_m, curr_c

In [ ]:
def evaluate_and_plot(data_x, data_y, fm, fc):
    y_true = np.array(data_y)
    y_pred = fm * data_x + fc

    # Accuracy Metrics
    r2 = 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))
    mae = np.mean(np.abs(y_true - y_pred))

    print("\nStep 7: Accuracy Metrics")
    print(f"  - R² Score: {r2:.4f}")
    print(f"  - Mean Absolute Error (MAE): {mae:.4f}")

    # Plotting
    plt.figure(figsize=(10, 6))
    plt.scatter(data_x, data_y, color='red', label='Actual Data')
    plt.plot(data_x, y_pred, color='blue', label=f'Model: y={fm:.2f}x + {fc:.2f}')
    plt.title('Linear Regression Fit (SymPy + Pandas)')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
cost_lambda = sp.lambdify((m, c, x, y), cost_func, "numpy")

def compute_total_cost(m_val, c_val, x_data, y_data):
    return np.mean(cost_lambda(m_val, c_val, x_data, y_data))


In [ ]:
def plot_cost_contour(data_x, data_y, fm, fc):
    m_range = np.linspace(fm - 2, fm + 2, 50)
    c_range = np.linspace(fc - 2, fc + 2, 50)
    M, C = np.meshgrid(m_range, c_range)

    Z = np.vectorize(lambda mv, cv: compute_total_cost(mv, cv, data_x, data_y))(M, C)

    plt.figure(figsize=(8, 6))
    cp = plt.contourf(M, C, Z, levels=20, cmap="viridis")
    plt.colorbar(cp, label="Cost (MSE)")
    plt.plot(fm, fc, "ro", label=f"Minimum (m={fm:.2f}, b={fc:.2f})")
    plt.title("Cost Function Contour Landscape")
    plt.xlabel("Slope (m)")
    plt.ylabel("Intercept (c)")
    plt.legend()
    plt.show()

In [ ]:
final_m, final_c = train_linear_regression(X_train, Y_train)
evaluate_and_plot(X_train, Y_train, final_m, final_c)
plot_cost_contour(X_train, Y_train, final_m, final_c)